In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# -----------------------------
# Load Dataset
# -----------------------------
dataset = pd.read_csv('/content/reddit_preprocessing.csv')
cleaned_dataset = dataset.dropna()

X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# -----------------------------
# Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42
)

# -----------------------------
# TF-IDF
# -----------------------------
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# -----------------------------
# Base Models
# -----------------------------
lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    class_weight='balanced',
    learning_rate=0.08,
    n_estimators=350,
    max_depth=10,
    random_state=42
)

logreg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial'
)

svc = LinearSVC(class_weight='balanced')

# -----------------------------
# Meta Model (XGBoost)
# -----------------------------
xgb_meta = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    learning_rate=0.1,
    n_estimators=200,
    max_depth=5,
    random_state=42,
    use_label_encoder=False
)

# -----------------------------
# Stacking Model
# -----------------------------
stacking_model = StackingClassifier(
    estimators=[
        ('lgbm', lgbm),
        ('lr', logreg),
        ('svc', svc)
    ],
    final_estimator=xgb_meta,
    cv=5,
    n_jobs=-1,
    passthrough=True   # 🔥 VERY IMPORTANT
)

# -----------------------------
# Train
# -----------------------------
stacking_model.fit(X_train_tfidf, y_train)

# -----------------------------
# Predict
# -----------------------------
y_pred = stacking_model.predict(X_test_tfidf)

# -----------------------------
# Evaluation
# -----------------------------
print(classification_report(y_test, y_pred))

KeyboardInterrupt: 